In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
language_role_schema = StructType(fields= [
                       StructField("roleId", IntegerType(), True),
                       StructField("languageRole", StringType(), True)
])

In [0]:
language_role_df = spark.read \
                   .schema(language_role_schema) \
                   .option("multiline", True) \
                   .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")

In [0]:
display(language_role_df)

roleId,languageRole
1,Original
2,Spoken


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
language_role_renamed_df = add_ingestion_date(language_role_df) \
                            .withColumnsRenamed({"roleId": "role_Id", "languageRole": "language_Role"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(language_role_renamed_df)

role_Id,language_Role,ingestion_date,environment,file_date
1,Original,2026-09-10T20:12:58.317255Z,production,2024-12-30
2,Spoken,2026-09-10T20:12:58.317255Z,production,2024-12-30


In [0]:
language_role_renamed_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.language_role")

In [0]:
display(spark.read.table("movie_silver.language_role"))

role_Id,language_Role,ingestion_date,environment,file_date
1,Original,2026-09-10T20:12:59.186205Z,production,2024-12-30
2,Spoken,2026-09-10T20:12:59.186205Z,production,2024-12-30


In [0]:
%sql
SELECT * FROM movie_silver.language_role;

role_Id,language_Role,ingestion_date,environment,file_date
1,Original,2026-09-10T20:12:59.186205Z,production,2024-12-30
2,Spoken,2026-09-10T20:12:59.186205Z,production,2024-12-30


In [0]:
dbutils.notebook.exit("Success")